In [ ]:
import os

# Render figure text with LaTeX (journalstyle "latex" style). Add latex to PATH here if it is missing
# os.environ["PATH"] = "/path/to/texlive/bin/x86_64-linux" + os.pathsep + os.environ["PATH"]

In [ ]:
from pathlib import Path

import journalstyle as js
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from src.gbp.hamiltonian import gbp_diag_numpy
from src.gbp.runs import (
    CQA_CACHE_PATH,
    QA_CACHE_PATH,
    REPO_ROOT,
    js_divergence,
    load_cqa_runs,
    load_qa_cqa_merged,
    load_qa_runs,
)
from src.gbp.search import search_problem
from src.run_cache import load_derived_cached

plt.style.use(["journal", "journal-aps", "latex"])

FIGS_DIR = Path("./figs/journal")
FIGS_DIR.mkdir(parents=True, exist_ok=True)
# The caches (cache/qa-gbp-runs.npz, cache/cqa-xy-gbp-runs.npz) are shared by all notebooks via src/gbp/runs.py.
# When to refresh: False = use the cache as-is / "new" = additionally load only uncached runs /
# True = reload all runs and rebuild (e.g. after changing the columns of the loaders in src/gbp/runs.py)
REFRESH_CACHE = False
# merged (QA x CQA join + distribution distances, cache/qa-cqa-gbp-merged.pkl) is rebuilt automatically when the run caches are updated.
# True = always rebuild (e.g. after changing the join logic in src/gbp/runs.py)
REFRESH_MERGED_CACHE = False
# Cache of derived data (for the D_JS vs r figure). Rebuilt automatically when the source run caches are updated.
# Set to True to rebuild after changing the parameters below or the contents of build_data
REFRESH_DERIVED = False
DATA_CACHE_PATH = REPO_ROOT / "cache" / "qa-cqa-penalty-ratio-data.pkl"

# Use only grid points with 6 per decade (1, 1.5, 2, 3, 5, 7 × 10^k) so points are evenly spaced on a log axis
# (same as log_grid in qa-gbp-add-tasks.py / cqa-xy-gbp-add-tasks.py). Other points in existing data (4, 6, 8, 9 × 10^k) are not used
LOG_GRID_MANTISSAS = [1, 1.5, 2, 3, 5, 7]


def log_grid(lo: float, hi: float) -> list[float]:
    values = {round(m * 10.0**k, 10) for k in range(-6, 7) for m in LOG_GRID_MANTISSAS}
    return sorted(v for v in values if lo <= v <= hi)


TIME_GRID = log_grid(1, 1e5)

N_LIST = [6, 8, 10, 12]
PENALTY_OFFSETS = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
T_LIST_MATCH = [1e3, 1e4, 1e5]  # T values for js-vs-penalty-ratio (b)

In [ ]:
import matplotlib as mpl


def cmap_colors(name: str, n: int, lo: float = 0.0, hi: float = 0.85) -> list[str]:
    # Take n evenly spaced colors from a continuous colormap (clip the too-light end at hi)
    return [mpl.colors.to_hex(c) for c in mpl.colormaps[name](np.linspace(lo, hi, n))]


THEME = {
    "offset_colors": cmap_colors("viridis", 7),
    "n_colors": ["#0077BB", "#EE7733", "#009988", "#CC3311"],
    "t_colors": ["#EE3377", "#33BBEE", "#555555"],
    "markers": ["o", "s", "^", "D", "v", "P", "X"],
    "linestyles": ["-"],  # Cycled per series
    "fill": "white",  # Marker fill: "closed" (same color as line) / "white" (hollow, white inside) / "none" (transparent inside)
    "capsize": 2,
    "marker_every": 3,  # Draw curves at all T points; thin out markers/error bars
}

# Line widths, marker sizes and font sizes follow the journalstyle (journal-aps) rcParams.
# Only marker edges and error bars are changed: journalstyle's lines.markeredgewidth (0.4 pt) makes hollow markers thin
# and falls below the APS minimum line width of 0.5 pt, so match the axis line width (axes.linewidth = 0.6 pt)
STROKE_LW = plt.rcParams["axes.linewidth"]


def series(theme: dict, kind: str, i: int) -> dict:
    """Color, marker and line style of the i-th series of kind ("offset" / "n" / "t")."""
    color = theme[f"{kind}_colors"][i]
    linestyles = theme["linestyles"]
    return {
        "color": color,
        "marker": theme["markers"][i],
        "ls": linestyles[i % len(linestyles)],
        "mfc": {"closed": color, "white": "white", "none": "none"}[theme["fill"]],
    }


def plot_series(ax, x, y, yerr, st: dict, theme: dict, label=None, size=1.0, lw_scale=1.0, markevery=None):
    """Draw a curve (with thinned-out markers) and error bars only at the marker positions. size and lw_scale are multipliers on the journalstyle values (for insets)."""
    x, y = np.asarray(x), np.asarray(y)
    every = theme["marker_every"] if markevery is None else markevery
    ax.plot(
        x, y, ls=st["ls"], color=st["color"], lw=plt.rcParams["lines.linewidth"] * lw_scale,
        marker=st["marker"], markevery=every, ms=plt.rcParams["lines.markersize"] * size, mew=STROKE_LW * size, mfc=st["mfc"],
        label=label,
    )
    if yerr is not None:
        sl = slice(None, None, every)
        ax.errorbar(
            x[sl], y[sl], yerr=np.asarray(yerr)[sl], fmt="none", ecolor=st["color"],
            capsize=theme["capsize"] * size, elinewidth=STROKE_LW * size, zorder=1.9,
        )


def log_x_ticks(ax):
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10, numticks=20))
    ax.xaxis.set_minor_locator(mticker.LogLocator(base=10, subs=np.arange(2, 10), numticks=20))

In [ ]:
df_qa_all = load_qa_runs(refresh=REFRESH_CACHE)
df_cqa_all = load_cqa_runs(refresh=REFRESH_CACHE)
# The merged cache is shared with other notebooks, so filter by T after joining
merged = load_qa_cqa_merged(df_qa_all, df_cqa_all, refresh=REFRESH_MERGED_CACHE)
merged = merged[merged["T"].isin(TIME_GRID)].reset_index(drop=True)
df_qa = df_qa_all[df_qa_all["T"].isin(TIME_GRID)].sort_values("run_id").reset_index(drop=True)
df_cqa = df_cqa_all[df_cqa_all["T"].isin(TIME_GRID)].sort_values("run_id").reset_index(drop=True)
print(f"qa-gbp: {df_qa.shape}, cqa-xy-gbp: {df_cqa.shape}, merged: {merged.shape}")

In [ ]:
summary_qa = (
    df_qa[df_qa["n_vertices"].isin(N_LIST)]
    .groupby(["penalty_offset", "n_vertices", "T"])[["optimal_probability", "js_p_uniform"]]
    .agg(["mean", "sem"])
)
summary_qa.columns = ["_".join(col) for col in summary_qa.columns]
summary_qa = summary_qa.reset_index()

INSET_N_VERTICES = [6, 8]
INSET_T_MIN = 1e2
INSET_BOUNDS = [0.30, 0.42, 0.66, 0.52]  # [x0, y0, width, height] in axes fraction of the parent panel


def plot_qa_by_n(theme: dict):
    width, height = js.figure_size("aps", columns=2, aspect=0.5)
    fig, axes = plt.subplots(2, len(N_LIST), figsize=(width, height), sharex=True, sharey="row")
    for col, n_vertices in enumerate(N_LIST):
        n_group = summary_qa[summary_qa["n_vertices"] == n_vertices]
        inset_ax = axes[1, col].inset_axes(INSET_BOUNDS) if n_vertices in INSET_N_VERTICES else None
        for i, penalty_offset in enumerate(PENALTY_OFFSETS):
            group = n_group[n_group["penalty_offset"] == penalty_offset].sort_values("T")
            st = series(theme, "offset", i)
            label = rf"$\mu_{{+}}={penalty_offset:g}$" if col == 0 else None
            plot_series(axes[0, col], group["T"], group["optimal_probability_mean"], group["optimal_probability_sem"], st, theme, label)
            plot_series(axes[1, col], group["T"], group["js_p_uniform_mean"], group["js_p_uniform_sem"], st, theme)
            if inset_ax is not None:
                zoom = group[group["T"] >= INSET_T_MIN]
                plot_series(inset_ax, zoom["T"], zoom["js_p_uniform_mean"], zoom["js_p_uniform_sem"], st, theme, size=0.8, lw_scale=0.78)
        if inset_ax is not None:
            inset_ax.set_xscale("log")
            log_x_ticks(inset_ax)
            inset_ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
            inset_ax.tick_params(labelsize=6, length=2, pad=1)
            inset_ax.tick_params(which="minor", length=1)
            inset_ax.yaxis.get_offset_text().set_fontsize(6)
        axes[0, col].set_xscale("log")
        axes[1, col].set_xscale("log")
        log_x_ticks(axes[1, col])
        axes[1, col].set_xlabel("$T$")
    axes[0, 0].set_ylabel(r"$P_{\mathrm{GS}}^{\mathrm{QA}}$")
    axes[1, 0].set_ylabel(r"$D_{\mathrm{JS}} \left(p_{\mathrm{QA}}, u\right)$")

    js.legend(axes, loc="above")
    js.label_panels(axes, journal="aps")
    return fig


fig = plot_qa_by_n(THEME)
fig.savefig(FIGS_DIR / "qa-gbp-success_probability_and_js_divergence_by_N.pdf")
plt.show()

In [ ]:
summary_cqa = (
    df_cqa[df_cqa["n_vertices"].isin(N_LIST) & (df_cqa["topology"] == "complete")]
    .groupby(["n_vertices", "T"])[["optimal_probability", "js_p_uniform"]]
    .agg(["mean", "sem"])
)
summary_cqa.columns = ["_".join(col) for col in summary_cqa.columns]
summary_cqa = summary_cqa.reset_index()


def plot_cqa_by_n(theme: dict):
    fig, axes = plt.subplots(2, 1, figsize=js.figure_size("aps", columns=1, aspect=0.95), sharex=True)
    for i, n_vertices in enumerate(N_LIST):
        group = summary_cqa[summary_cqa["n_vertices"] == n_vertices].sort_values("T")
        st = series(theme, "n", i)
        for ax, column in zip(axes, ["optimal_probability", "js_p_uniform"]):
            label = f"$N={n_vertices}$" if ax is axes[0] else None
            plot_series(ax, group["T"], group[f"{column}_mean"], group[f"{column}_sem"], st, theme, label)
    for ax in axes:
        ax.set_xscale("log")
    axes[1].set_xlabel("$T$")
    axes[0].set_ylabel(r"$P_{\mathrm{GS}}^{\mathrm{CQA}}$")
    axes[1].set_ylabel(r"$D_{\mathrm{JS}} \left(p_{\mathrm{CQA}}, u\right)$")
    axes[0].legend(loc="lower right", ncol=2)

    js.label_panels(axes, journal="aps")
    return fig


fig = plot_cqa_by_n(THEME)
fig.savefig(FIGS_DIR / "cqa-xy-gbp-success_probability_and_js_divergence_by_N.pdf")
plt.show()

In [ ]:
# Mean ± SEM over seeds
js_stats = (
    merged[(merged["topology"] == "complete") & merged["penalty_offset"].isin(PENALTY_OFFSETS)]
    .groupby(["n_vertices", "penalty_offset", "T"])["js"]
    .agg(["mean", "sem"])
    .reset_index()
)

JS_INSET_N_VERTICES = [6]  # N with an inset (range and position shared with the QA figure: INSET_T_MIN, INSET_BOUNDS)


def plot_js_qa_cqa_by_n(theme: dict):
    fig, axes = plt.subplots(1, len(N_LIST), figsize=js.figure_size("aps", columns=2, aspect=0.31), sharey=True)
    for ax, n in zip(axes, N_LIST):
        inset_ax = ax.inset_axes(INSET_BOUNDS) if n in JS_INSET_N_VERTICES else None
        for i, offset in enumerate(PENALTY_OFFSETS):
            group = js_stats[(js_stats["n_vertices"] == n) & (js_stats["penalty_offset"] == offset)].sort_values("T")
            st = series(theme, "offset", i)
            label = rf"$\mu_{{+}}={offset:g}$" if ax is axes[0] else None
            plot_series(ax, group["T"], group["mean"], group["sem"], st, theme, label)
            if inset_ax is not None:
                zoom = group[group["T"] >= INSET_T_MIN]
                plot_series(inset_ax, zoom["T"], zoom["mean"], zoom["sem"], st, theme, size=0.8, lw_scale=0.78)
        if inset_ax is not None:
            inset_ax.set_xscale("log")
            log_x_ticks(inset_ax)
            inset_ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
            inset_ax.tick_params(labelsize=6, length=2, pad=1)
            inset_ax.tick_params(which="minor", length=1)
            inset_ax.yaxis.get_offset_text().set_fontsize(6)
        ax.set_xscale("log")
        ax.set_xlabel("$T$")
        log_x_ticks(ax)
    axes[0].set_ylabel(r"$D_{\mathrm{JS}} \left(p_{\mathrm{QA}}, p_{\mathrm{CQA}}\right)$")

    js.legend(axes[0], loc="above")
    js.label_panels(axes, journal="aps")
    return fig


fig = plot_js_qa_cqa_by_n(THEME)
fig.savefig(FIGS_DIR / "js-qa-cqa-by-N.pdf")
plt.show()

In [ ]:
TOPOLOGY = "complete"
# Instance generation parameters (same as qa-gbp-add-tasks.py). Used to reconstruct W from the seed
EDGE_PROB = 0.8
MIN_DEGENERATE = 4
MAX_SEED_TRIES = 1000

PR_TIME_GRID = log_grid(10, 1e5)
PENALTY_OFFSET_GRID = log_grid(0.01, 10)


def instance_data(n: int, seed: int) -> dict:
    _, _, W, _, _, mu_opt, _, _ = search_problem(
        n, EDGE_PROB, None, seed, PENALTY_OFFSET_GRID[0], MIN_DEGENERATE, MAX_SEED_TRIES, build_qutip=False
    )
    diag_c, diag_p = gbp_diag_numpy(W, n)
    feasible = diag_p < 1e-10
    e0 = diag_c[feasible].min()
    gs = np.where(feasible & (np.abs(diag_c - e0) < 1e-10))[0]
    flips = gs[:, None] ^ (1 << np.arange(n))[None, :]  # (ground state, flip position) -> intermediate state m
    return {"n": n, "mu_opt": mu_opt, "diag_c": diag_c, "e0": e0, "gs": gs, "flips": flips, "dE": diag_c[flips] - e0}


def _pt_matrices(f: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Return the projection U onto the even sector, P_g H_XY P_g in the even sector, and second-order paths (a, b, ΔE_obj(m))
    n, gs, diag_c, e0 = f["n"], f["gs"], f["diag_c"], f["e0"]
    index = {int(g): k for k, g in enumerate(gs)}
    full = (1 << n) - 1
    _, label = np.unique(np.minimum(gs, gs ^ full), return_inverse=True)
    U = np.zeros((len(gs), label.max() + 1))
    U[np.arange(len(gs)), label] = 1 / np.sqrt(2)

    H_xy = np.zeros((len(gs), len(gs)))
    paths = []
    for a, g in enumerate(gs):
        g = int(g)
        for i in range(n):
            m = g ^ (1 << i)
            for j in range(n):
                b = index.get(m ^ (1 << j))
                if b is not None:
                    paths.append((a, b, diag_c[m] - e0))
            for j in range(i + 1, n):
                if ((g >> i) ^ (g >> j)) & 1:
                    b = index.get(g ^ (1 << i) ^ (1 << j))
                    if b is not None:
                        H_xy[b, a] -= 2  # <b|(σ^xσ^x + σ^yσ^y)|a> = 2; H_XY has a negative sign
    return U, U.T @ H_xy @ U, np.array(paths)


def _ground_distribution(H: np.ndarray) -> tuple[np.ndarray, float]:
    w, v = np.linalg.eigh(H)
    p = v[:, 0] ** 2
    return p / p.sum(), w[1] - w[0] if len(w) > 1 else np.inf


def build_data() -> pd.DataFrame:
    merged = load_qa_cqa_merged(load_qa_runs(), load_cqa_runs())
    merged = merged[
        (merged["topology"] == TOPOLOGY)
        & merged["n_vertices"].isin(N_LIST)
        & merged["T"].isin(PR_TIME_GRID)
        & merged["penalty_offset"].isin(PENALTY_OFFSET_GRID)
    ].reset_index(drop=True)
    merged = merged[["n_vertices", "seed", "T", "penalty_offset", "optimal_probability_qa", "optimal_probability_cqa", "js"]]

    instances = {
        (n, seed): instance_data(int(n), int(seed))
        for n, seed in merged[["n_vertices", "seed"]].drop_duplicates().itertuples(index=False)
    }
    inst_df = pd.DataFrame(
        [
            {"n_vertices": n, "seed": seed, "mu_opt": f["mu_opt"], "dE_max": np.abs(f["dE"]).max(), "n_gs": len(f["gs"])}
            for (n, seed), f in instances.items()
        ]
    )

    pt_rows = []
    for (n, seed), f in instances.items():
        U, H_xy_even, paths = _pt_matrices(f)
        p_cqa, gap_xy = _ground_distribution(H_xy_even)
        a, b, dE = paths[:, 0].astype(int), paths[:, 1].astype(int), paths[:, 2]
        for offset in PENALTY_OFFSET_GRID:
            mu = f["mu_opt"] + offset
            H2 = np.zeros((len(f["gs"]), len(f["gs"])))
            np.add.at(H2, (b, a), -1 / (4 * mu + dE))
            p_qa, _ = _ground_distribution(U.T @ H2 @ U)
            pt_rows.append(
                {"n_vertices": n, "seed": seed, "penalty_offset": offset, "gap_xy": gap_xy, "js_pt": js_divergence(p_qa, p_cqa)}
            )
    pt_df = pd.DataFrame(pt_rows)

    data = merged.merge(inst_df, on=["n_vertices", "seed"]).merge(pt_df, on=["n_vertices", "seed", "penalty_offset"])
    data["mu"] = data["mu_opt"] + data["penalty_offset"]
    data["r"] = 4 * data["mu"] / data["dE_max"]
    return data


data = load_derived_cached(DATA_CACHE_PATH, build_data, depends_on=[QA_CACHE_PATH, CQA_CACHE_PATH], refresh=REFRESH_DERIVED)

# Thresholds (same as Figs. 4 and 5 of the manuscript). Points with D_JS <= JS_FLOOR are where the QA and CQA distributions match exactly by symmetry;
# they cannot be shown on a log axis, so they are excluded from (a) (but included in the agreement fraction in (b))
P_GS_MIN = 0.99
JS_MAX = 0.01
JS_FLOOR = 1e-10
T_LONG = 1e5
N_MAIN = 12
R_FIT_MIN = 1  # The regression line in (a) is fit on r >= R_FIT_MIN, excluding the flat region

data["match"] = (
    (data["optimal_probability_qa"] > P_GS_MIN) & (data["optimal_probability_cqa"] > P_GS_MIN) & (data["js"] < JS_MAX)
)
long_T = data[data["T"] == T_LONG]
nonzero = long_T[long_T["js"] > JS_FLOOR]

In [ ]:
def _binned(df: pd.DataFrame, bins: np.ndarray, column: str, agg: str, min_count: int = 10) -> tuple:
    # Aggregate per log r bin (drop bins with fewer than min_count points). Bin centers are geometric means
    g = df.groupby(pd.cut(df["r"], bins), observed=True)[column]
    value, count = g.agg(agg), g.size()
    keep = count >= min_count
    centers = np.array([np.sqrt(iv.left * iv.right) for iv in value.index[keep]])
    return centers, value[keep].to_numpy(), count[keep].to_numpy()


def fit_per_instance(df: pd.DataFrame) -> tuple[float, float]:
    # Regress log D_JS on log r (OLS) per instance and take the median slope.
    # Place the line through the centroid (mean of logs) of the points with r >= R_FIT_MIN
    d = df[df["r"] >= R_FIT_MIN]
    slopes = [np.polyfit(np.log10(g["r"]), np.log10(g["js"]), 1)[0] for _, g in d.groupby(["n_vertices", "seed"])]
    slope = np.median(slopes)
    return slope, np.log10(d["js"]).mean() - slope * np.log10(d["r"]).mean()


R_LABEL = r"$r = 4\mu / \max|\Delta E_{\mathrm{obj}}|$"
FITS = {n: fit_per_instance(nonzero[nonzero["n_vertices"] == n]) for n in N_LIST}


def plot_js_vs_penalty_ratio(theme: dict):
    # Share the r axis between (a) and (b) to align their ranges
    fig, axes = plt.subplots(2, 1, sharex=True, figsize=js.figure_size("aps", columns=1, aspect=1.4), constrained_layout=True)

    # (a)
    ax = axes[0]

    for i, n in enumerate(N_LIST):
        d = nonzero[nonzero["n_vertices"] == n]
        ax.scatter(d["r"], d["js"], s=2.5, color="gray", alpha=0.2, linewidths=0, rasterized=True)
        # ax.scatter(d["r"], d["js"], s=2.5, color=series(theme, "n", i)["color"], alpha=0.2, linewidths=0, rasterized=True)
    for i, n in enumerate(N_LIST):
        slope, intercept = FITS[n]
        x = np.logspace(np.log10(R_FIT_MIN), np.log10(nonzero.loc[nonzero["n_vertices"] == n, "r"].max()), 50)
        plot_series(ax, x, 10 ** (intercept + slope * np.log10(x)), None, series(theme, "n", i), theme, f"$N={n}$: ${slope:.2f}$", markevery=7)

    r_guide = np.logspace(0, 1.6, 10)
    ax.plot(r_guide, 3e-3 * r_guide**-2.0, "k--", lw=0.8)
    ax.text(8, 3e-3 * 8**-2 * 3, r"$\propto r^{-2}$")
    ax.axvline(1, color="0.5", ls=":", lw=0.8)
    ax.set(xscale="log", yscale="log", ylabel=r"$D_{\mathrm{JS}}(p_{\mathrm{QA}}, p_{\mathrm{CQA}})$", ylim=(1e-9, 1))
    ax.legend(loc="lower left")

    # (b)
    ax = axes[1]
    for i, T in enumerate(T_LIST_MATCH):
        d = data[(data["n_vertices"] == N_MAIN) & (data["T"] == T)]
        x, frac, count = _binned(d, np.logspace(-0.7, 1.5, 12), "match", "mean")
        plot_series(
            ax, x, frac, np.sqrt(frac * (1 - frac) / count), series(theme, "t", i), theme,
            f"$T=10^{{{int(np.log10(T))}}}$", markevery=1,
        )
    ax.axvline(1, color="0.5", ls=":", lw=0.8)
    ax.set(xscale="log", xlabel=R_LABEL, ylabel=r"$f_{\mathrm{match}}$", ylim=(-0.03, 1.03))
    ax.legend(loc="lower left")

    js.label_panels(axes, journal="aps", inside=True)
    return fig


fig = plot_js_vs_penalty_ratio(THEME)
fig.savefig(FIGS_DIR / "js-vs-penalty-ratio.pdf")
plt.show()